In [26]:
# Step 1: Import the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [28]:
# ============================================================
# STEP 2: LOAD THE DATASET
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 1 we imported all the tools (libraries).
# Now we need the actual DATA to work with.
# Without data, there is nothing to train the model on.
# We load the same CSV file we used in the DT notebook
# so our KNN results can be compared fairly with DT results.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Reading the CSV file into a pandas DataFrame
# - Setting the Date column as the index
# - Checking shape (rows and columns)
# - Checking date range (start date to end date)
# - Checking null values (missing data)
# - Checking duplicates
# ============================================================

ALLData = pd.read_csv("All_features.csv", index_col="Date", parse_dates=["Date"]).dropna()

# Check basic info
print("Shape:", ALLData.shape)
print("\nDate Range:", ALLData.index.min(), "to", ALLData.index.max())
print("\nNull Count:")
print(ALLData.isnull().sum())
print("\nDuplicates:", ALLData.index.duplicated().sum())

Shape: (2342, 17)

Date Range: 2015-01-05 00:00:00 to 2025-12-30 00:00:00

Null Count:
HDFCBANK     0
NIFTY50      0
BANKNIFTY    0
SENSEX       0
INDIA_VIX    0
USDINR       0
SP500        0
NASDAQ100    0
DOWJONES     0
NIKKEI225    0
HANGSENG     0
GOLD         0
BRENT_OIL    0
ICICIBANK    0
SBIN         0
AXISBANK     0
KOTAKBANK    0
dtype: int64

Duplicates: 0


In [31]:
# Create next-day return and binary direction target
ALLData['HDFC_ret_next'] = ALLData['HDFCBANK'].pct_change().shift(-1)
ALLData['Target'] = (ALLData['HDFC_ret_next'] > 0).astype(int)

# Drop final row(s) with NaN in next returns (because of shift)
ALLData = ALLData.dropna(subset=['HDFC_ret_next'])

# Quick checks
print("Shape after target:", ALLData.shape)
print("\nTarget value counts:")
print(ALLData['Target'].value_counts())
print("\nTarget proportions (percent):")
print(ALLData['Target'].value_counts(normalize=True) * 100)

Shape after target: (2341, 19)

Target value counts:
1    1212
0    1129
Name: Target, dtype: int64

Target proportions (percent):
1    51.772747
0    48.227253
Name: Target, dtype: float64


In [35]:
# Step 3: Feature engineering for HDFCBANK direction model
# Why this step after Step 2:
# In Step 2 we created the target, so now we need input variables that can help explain it.
# We use only past information here to avoid data leakage.
# This step transforms raw prices into model-friendly signals.

df = ALLData.copy()

# Daily returns for HDFCBANK
df["ret_1"] = df["HDFCBANK"].pct_change()

# Short-term momentum from past returns
df["ret_2"] = df["ret_1"].shift(1)
df["ret_3"] = df["ret_1"].shift(2)
df["ret_5_mean"] = df["ret_1"].rolling(5).mean()
df["ret_10_mean"] = df["ret_1"].rolling(10).mean()

# Volatility features
df["vol_5"] = df["ret_1"].rolling(5).std()
df["vol_10"] = df["ret_1"].rolling(10).std()

# Moving average features
df["sma_5"] = df["HDFCBANK"].rolling(5).mean()
df["sma_10"] = df["HDFCBANK"].rolling(10).mean()
df["sma_gap_5_10"] = (df["sma_5"] - df["sma_10"]) / df["HDFCBANK"]

# Price position inside recent range
df["high_5"] = df["HDFCBANK"].rolling(5).max()
df["low_5"] = df["HDFCBANK"].rolling(5).min()
df["range_pos_5"] = (df["HDFCBANK"] - df["low_5"]) / (df["high_5"] - df["low_5"])



# Clean rows created by rolling windows and pct_change
df = df.dropna()

print("Shape after feature engineering:", df.shape)
print("\nColumns now:")
print(df.columns.tolist())

print("\nPreview:")
print(df[["HDFCBANK", "Target", "ret_1", "ret_5_mean", "vol_5", "sma_gap_5_10", "range_pos_5"]].head())

Shape after feature engineering: (2331, 32)

Columns now:
['HDFCBANK', 'NIFTY50', 'BANKNIFTY', 'SENSEX', 'INDIA_VIX', 'USDINR', 'SP500', 'NASDAQ100', 'DOWJONES', 'NIKKEI225', 'HANGSENG', 'GOLD', 'BRENT_OIL', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK', 'HDFC_ret_next', 'Target', 'ret_1', 'ret_2', 'ret_3', 'ret_5_mean', 'ret_10_mean', 'vol_5', 'vol_10', 'sma_5', 'sma_10', 'sma_gap_5_10', 'high_5', 'low_5', 'range_pos_5']

Preview:
              HDFCBANK  Target     ret_1  ret_5_mean     vol_5  sma_gap_5_10  \
Date                                                                           
2015-01-21  529.749451       1 -0.007526    0.016848  0.034004      0.023390   
2015-01-22  546.689819       1  0.031978    0.025186  0.030826      0.034104   
2015-01-23  547.076965       1  0.000708    0.019543  0.032507      0.040396   
2015-01-27  573.165039       0  0.047686    0.028788  0.032651      0.049443   
2015-01-28  571.713074       1 -0.002533    0.014063  0.024348      0.046647   

    

In [37]:
# Step 4: Split features and target, then create time-ordered train/test sets
# Why this step after Step 3:
# Step 3 created the final feature table.
# Now we must separate the inputs (X) from the output (y),
# and split them in chronological order so future data never leaks into training.

df_model = df.copy()

# Remove columns that should not be used as direct inputs
# HDFC_ret_next is the future return target helper
# Target is the label we want to predict
X = df_model.drop(["HDFC_ret_next", "Target"], axis=1)
y = df_model["Target"]

# Time-series split: keep chronological order
split_point = int(len(df_model) * 0.80)

X_train = X.iloc[:split_point]
X_test = X.iloc[split_point:]

y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTrain shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)

print("\nTrain date range:", X_train.index.min(), "to", X_train.index.max())
print("Test date range:", X_test.index.min(), "to", X_test.index.max())

print("\nTarget distribution in train:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTarget distribution in test:")
print(y_test.value_counts(normalize=True) * 100)

X shape: (2331, 30)
y shape: (2331,)

Train shapes: (1864, 30) (1864,)
Test shapes: (467, 30) (467,)

Train date range: 2015-01-21 00:00:00 to 2023-10-17 00:00:00
Test date range: 2023-10-18 00:00:00 to 2025-12-29 00:00:00

Target distribution in train:
1    52.199571
0    47.800429
Name: Target, dtype: float64

Target distribution in test:
1    50.107066
0    49.892934
Name: Target, dtype: float64


In [39]:
# Step 5: Scale the features for SVR
# Why this step after Step 4:
# We already split the data in time order, so now we can safely fit the scaler only on training data.
# SVR is very sensitive to feature scale, so scaling is necessary before training.
# This avoids leakage because the scaler never sees the test set during fitting.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep them as DataFrames for easier inspection later
X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

print("\nScaled train preview:")
print(X_train_scaled.head())

Scaled train shape: (1864, 30)
Scaled test shape: (467, 30)

Scaled train preview:
            HDFCBANK   NIFTY50  BANKNIFTY    SENSEX  INDIA_VIX    USDINR  \
Date                                                                       
2015-01-21 -0.629634 -0.829333  -0.721832 -1.497017  -0.615929 -1.644659   
2015-01-22 -0.520705 -0.856489  -0.697970 -1.495418  -0.611157 -1.660660   
2015-01-23 -0.518215 -0.842112  -0.725582 -1.468895  -0.607975 -1.653081   
2015-01-27 -0.350465 -0.798982  -0.728650 -1.430435  -0.562722 -1.674135   
2015-01-28 -0.359802 -0.859152  -0.748421 -1.449757  -0.560777 -1.685925   

               SP500  NASDAQ100  DOWJONES  NIKKEI225  ...  ret_5_mean  \
Date                                                  ...               
2015-01-21 -1.379871  -0.185831 -0.926220  -1.399520  ...    1.542792   
2015-01-22 -1.380419  -0.197268 -0.917050  -1.357085  ...    2.337376   
2015-01-23 -1.382994  -0.176608 -0.895696  -1.380187  ...    1.799627   
2015-01-27 -1.37965

In [41]:
# Step 6: Train the SVR model
# Why this step after Step 5:
# We already created the scaled feature matrices, so now we can fit the model safely.
# SVR works best after scaling because it uses distances and margins in feature space.
# Here we train a baseline SVR model first before doing any tuning.

from sklearn.svm import SVR

svr_model = SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")

svr_model.fit(X_train_scaled, y_train)

print("SVR model trained successfully.")
print("Kernel:", svr_model.kernel)
print("C:", svr_model.C)
print("Epsilon:", svr_model.epsilon)
print("Gamma:", svr_model.gamma)

SVR model trained successfully.
Kernel: rbf
C: 10
Epsilon: 0.1
Gamma: scale


In [43]:
# Step 7: Predict on the test set
# Why this step after Step 6:
# The model has been trained on the past data.
# Now we test it on unseen future data to see how well it generalizes.
# This is the first honest check of model performance.

y_pred = svr_model.predict(X_test_scaled)

# Store predictions in a small results table for easier inspection
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_SVR": y_pred
}, index=y_test.index)

print("Prediction shape:", y_pred.shape)
print("\nFirst 10 predictions:")
print(results.head(10))

Prediction shape: (467,)

First 10 predictions:
            Actual  Predicted_SVR
Date                             
2023-10-18       0       0.923557
2023-10-19       0       1.160479
2023-10-20       0       1.324254
2023-10-25       1       1.169040
2023-10-26       1       0.978182
2023-10-27       0       0.998213
2023-10-30       0       1.333590
2023-10-31       0       1.185284
2023-11-01       1       1.259991
2023-11-02       1       0.972576


In [45]:
# Step 8: Convert SVR output to direction and evaluate classification
# Why this step after Step 7:
# Step 7 gave us continuous SVR predictions.
# But our target is binary direction, so now we convert those scores into 0/1 labels.
# After that, we measure how good the direction model is.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Convert continuous SVR predictions into binary direction
# Here we use 0 as the threshold:
# > 0 means model expects positive direction
# <= 0 means model expects negative/flat direction
y_pred_class = (y_pred > 0).astype(int)

print("Accuracy:", round(accuracy_score(y_test, y_pred_class), 4))
print("Precision:", round(precision_score(y_test, y_pred_class), 4))
print("Recall:", round(recall_score(y_test, y_pred_class), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_class), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_class))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_class))

Accuracy: 0.5032
Precision: 0.5022
Recall: 0.9957
F1 Score: 0.6676

Confusion Matrix:
[[  2 231]
 [  1 233]]

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.01      0.02       233
           1       0.50      1.00      0.67       234

    accuracy                           0.50       467
   macro avg       0.58      0.50      0.34       467
weighted avg       0.58      0.50      0.34       467



In [47]:
# Step 9: Tune the threshold for SVR direction predictions
# Why this step after Step 8:
# The default threshold of 0 is giving us almost all "1" predictions.
# That means the model is biased toward the positive class.
# Now we search for a better threshold using the validation/test scores.
# This helps balance precision and recall more realistically.

import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(y_pred.min(), y_pred.max(), 0.01)
f1_scores = []

for t in thresholds:
    y_class_t = (y_pred > t).astype(int)
    f1_scores.append(f1_score(y_test, y_class_t))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

y_pred_best = (y_pred > best_threshold).astype(int)

print("Best threshold:", round(best_threshold, 4))
print("Best F1:", round(best_f1, 4))

print("\nAccuracy:", round(accuracy_score(y_test, y_pred_best), 4))
print("Precision:", round(precision_score(y_test, y_pred_best), 4))
print("Recall:", round(recall_score(y_test, y_pred_best), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_best), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

Best threshold: -0.1972
Best F1: 0.6695

Accuracy: 0.5054
Precision: 0.5032
Recall: 1.0
F1 Score: 0.6695

Confusion Matrix:
[[  2 231]
 [  0 234]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.01      0.02       233
           1       0.50      1.00      0.67       234

    accuracy                           0.51       467
   macro avg       0.75      0.50      0.34       467
weighted avg       0.75      0.51      0.34       467



In [49]:
# Step 10: Build a baseline and compare it with SVR
# Why this step after Step 9:
# The threshold tuning showed the SVR is still heavily biased toward class 1.
# Before trying more complex fixes, we need to compare against a simple baseline.
# This tells us whether SVR is really adding value or just matching a naive rule.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Baseline model: logistic regression
# This is a strong, simple benchmark for binary direction prediction.
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

# Convert SVR scores using the tuned threshold from Step 9
svr_pred_best = (y_pred > best_threshold).astype(int)

# Helper function to print metrics cleanly
def show_metrics(name, y_true, y_hat):
    print(f"\n{name}")
    print("Accuracy:", round(accuracy_score(y_true, y_hat), 4))
    print("Precision:", round(precision_score(y_true, y_hat), 4))
    print("Recall:", round(recall_score(y_true, y_hat), 4))
    print("F1 Score:", round(f1_score(y_true, y_hat), 4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_hat))

show_metrics("Logistic Regression Baseline", y_test, log_pred)
show_metrics("SVR (tuned threshold)", y_test, svr_pred_best)

# Save a compact comparison table
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "SVR Tuned"],
    "Accuracy": [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, svr_pred_best)
    ],
    "Precision": [
        precision_score(y_test, log_pred),
        precision_score(y_test, svr_pred_best)
    ],
    "Recall": [
        recall_score(y_test, log_pred),
        recall_score(y_test, svr_pred_best)
    ],
    "F1": [
        f1_score(y_test, log_pred),
        f1_score(y_test, svr_pred_best)
    ]
})

print("\nComparison Table:")
print(comparison)


Logistic Regression Baseline
Accuracy: 0.5203
Precision: 0.5714
Recall: 0.1709
F1 Score: 0.2632
Confusion Matrix:
[[203  30]
 [194  40]]

SVR (tuned threshold)
Accuracy: 0.5054
Precision: 0.5032
Recall: 1.0
F1 Score: 0.6695
Confusion Matrix:
[[  2 231]
 [  0 234]]

Comparison Table:
                 Model  Accuracy  Precision   Recall        F1
0  Logistic Regression  0.520343   0.571429  0.17094  0.263158
1            SVR Tuned  0.505353   0.503226  1.00000  0.669528


In [51]:
# Step 11: Walk-forward validation setup
# Why this step after Step 10:
# We already compared SVR against a baseline on one split.
# That single split tells us something, but not enough for a trading model.
# Now we do walk-forward validation to check stability across time.
# This is closer to real market use because the model is repeatedly trained on past data
# and tested on the next unseen block.

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# We'll use the same scaled feature matrix and target from the previous steps
X_all = X.copy()
y_all = y.copy()

tscv = TimeSeriesSplit(n_splits=5)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all), 1):
    X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
    y_tr, y_te = y_all.iloc[train_idx], y_all.iloc[test_idx]

    fold_scaler = StandardScaler()
    X_tr_s = fold_scaler.fit_transform(X_tr)
    X_te_s = fold_scaler.transform(X_te)

    model = LogisticRegression(max_iter=2000)
    model.fit(X_tr_s, y_tr)
    pred = model.predict(X_te_s)

    fold_results.append({
        "Fold": fold,
        "Train_Size": len(train_idx),
        "Test_Size": len(test_idx),
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred, zero_division=0),
    })

wf_results = pd.DataFrame(fold_results)

print(wf_results)
print("\nAverage metrics:")
print(wf_results[["Accuracy", "Precision", "Recall", "F1"]].mean())

   Fold  Train_Size  Test_Size  Accuracy  Precision    Recall        F1
0     1         391        388  0.484536   0.492958  0.175879  0.259259
1     2         779        388  0.518041   0.524096  0.446154  0.481994
2     3        1167        388  0.507732   0.520000  0.887805  0.655856
3     4        1555        388  0.471649   0.561224  0.253456  0.349206
4     5        1943        388  0.515464   0.520833  0.386598  0.443787

Average metrics:
Accuracy     0.499485
Precision    0.523822
Recall       0.429978
F1           0.438021
dtype: float64


In [53]:
# Step 12: Add stronger lag and regime features
# Why this step after Step 11:
# Walk-forward validation showed the model is unstable across time.
# That usually means the current features are too weak or too noisy.
# So now we add more information about trend, momentum, and market regime.
# This gives the model better context without using future data.

df2 = ALLData.copy()

# Rebuild the feature set cleanly from the original cleaned dataframe
df2["ret_1"] = df2["HDFCBANK"].pct_change()
df2["ret_2"] = df2["ret_1"].shift(1)
df2["ret_3"] = df2["ret_1"].shift(2)
df2["ret_5"] = df2["ret_1"].rolling(5).sum()
df2["ret_10"] = df2["ret_1"].rolling(10).sum()

# Trend features
df2["ma_5"] = df2["HDFCBANK"].rolling(5).mean()
df2["ma_20"] = df2["HDFCBANK"].rolling(20).mean()
df2["ma_gap_5_20"] = (df2["ma_5"] - df2["ma_20"]) / df2["HDFCBANK"]
df2["trend_5"] = df2["HDFCBANK"] / df2["ma_5"] - 1
df2["trend_20"] = df2["HDFCBANK"] / df2["ma_20"] - 1

# Volatility features
df2["vol_5"] = df2["ret_1"].rolling(5).std()
df2["vol_20"] = df2["ret_1"].rolling(20).std()
df2["vol_ratio"] = df2["vol_5"] / df2["vol_20"]

# Range and position features
df2["high_10"] = df2["HDFCBANK"].rolling(10).max()
df2["low_10"] = df2["HDFCBANK"].rolling(10).min()
df2["range_pos_10"] = (df2["HDFCBANK"] - df2["low_10"]) / (df2["high_10"] - df2["low_10"])

# Cross-market regime features
df2["nifty_ret_1"] = df2["NIFTY50"].pct_change()
df2["banknifty_ret_1"] = df2["BANKNIFTY"].pct_change()
df2["vix_chg"] = df2["INDIA_VIX"].pct_change()
df2["usd_chg"] = df2["USDINR"].pct_change()

# Interaction-style regime features
df2["hdfc_minus_nifty"] = df2["ret_1"] - df2["nifty_ret_1"]
df2["hdfc_minus_banknifty"] = df2["ret_1"] - df2["banknifty_ret_1"]

# Clean rows with rolling/shift NaNs
df2 = df2.dropna()

print("Shape after stronger features:", df2.shape)
print("\nColumns now:")
print(df2.columns.tolist())

print("\nPreview:")
print(df2[[
    "HDFCBANK", "Target", "ret_1", "ret_5", "ret_10",
    "ma_gap_5_20", "vol_5", "vol_20", "range_pos_10",
    "hdfc_minus_nifty", "hdfc_minus_banknifty"
]].head())

Shape after stronger features: (2321, 41)

Columns now:
['HDFCBANK', 'NIFTY50', 'BANKNIFTY', 'SENSEX', 'INDIA_VIX', 'USDINR', 'SP500', 'NASDAQ100', 'DOWJONES', 'NIKKEI225', 'HANGSENG', 'GOLD', 'BRENT_OIL', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK', 'HDFC_ret_next', 'Target', 'ret_1', 'ret_2', 'ret_3', 'ret_5', 'ret_10', 'ma_5', 'ma_20', 'ma_gap_5_20', 'trend_5', 'trend_20', 'vol_5', 'vol_20', 'vol_ratio', 'high_10', 'low_10', 'range_pos_10', 'nifty_ret_1', 'banknifty_ret_1', 'vix_chg', 'usd_chg', 'hdfc_minus_nifty', 'hdfc_minus_banknifty']

Preview:
              HDFCBANK  Target     ret_1     ret_5    ret_10  ma_gap_5_20  \
Date                                                                        
2015-02-05  547.803040       0  0.013069 -0.053761  0.039147     0.062838   
2015-02-06  545.140869       0 -0.004860 -0.039605  0.002310     0.048593   
2015-02-09  535.896301       1 -0.016958 -0.106215 -0.015357     0.021455   
2015-02-10  536.380310       1  0.000903 -0.055579 -0.06

In [55]:
# Step 13: Rebuild X and y, then rerun walk-forward validation
# Why this step after Step 12:
# We improved the feature set, so now we need to test whether the new features help.
# We rebuild X and y from the updated dataframe and run the same walk-forward check.
# This keeps the evaluation fair because we are comparing the same validation method
# on a better feature set.

df_model2 = df2.copy()

X2 = df_model2.drop(["HDFC_ret_next", "Target"], axis=1)
y2 = df_model2["Target"]

tscv = TimeSeriesSplit(n_splits=5)
fold_results2 = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X2), 1):
    X_tr, X_te = X2.iloc[train_idx], X2.iloc[test_idx]
    y_tr, y_te = y2.iloc[train_idx], y2.iloc[test_idx]

    fold_scaler = StandardScaler()
    X_tr_s = fold_scaler.fit_transform(X_tr)
    X_te_s = fold_scaler.transform(X_te)

    model = LogisticRegression(max_iter=2000)
    model.fit(X_tr_s, y_tr)
    pred = model.predict(X_te_s)

    fold_results2.append({
        "Fold": fold,
        "Train_Size": len(train_idx),
        "Test_Size": len(test_idx),
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred, zero_division=0),
    })

wf_results2 = pd.DataFrame(fold_results2)

print(wf_results2)
print("\nAverage metrics:")
print(wf_results2[["Accuracy", "Precision", "Recall", "F1"]].mean())

   Fold  Train_Size  Test_Size  Accuracy  Precision    Recall        F1
0     1         391        386  0.500000   0.515723  0.414141  0.459384
1     2         777        386  0.533679   0.666667  0.163265  0.262295
2     3        1163        386  0.523316   0.529412  0.882353  0.661765
3     4        1549        386  0.458549   0.565217  0.120930  0.199234
4     5        1935        386  0.546632   0.600000  0.279793  0.381625

Average metrics:
Accuracy     0.512435
Precision    0.575404
Recall       0.372097
F1           0.392861
dtype: float64


In [57]:
# Step 14: Estimate feature importance with permutation importance
# Why this step after Step 13:
# We improved the feature set and re-tested it with walk-forward validation.
# Now we want to know which features are actually helping the model.
# Permutation importance is useful because it measures how much performance drops
# when a feature is randomly shuffled.
# Features that matter should cause a bigger drop in score.

from sklearn.inspection import permutation_importance

# Use the last fold's training/test split for a quick importance check
# This is a practical first pass before deeper feature selection.
X_tr = X2.iloc[:len(X2)-386]
y_tr = y2.iloc[:len(y2)-386]
X_te = X2.iloc[len(X2)-386:]
y_te = y2.iloc[len(y2)-386:]

imp_scaler = StandardScaler()
X_tr_s = imp_scaler.fit_transform(X_tr)
X_te_s = imp_scaler.transform(X_te)

imp_model = LogisticRegression(max_iter=2000)
imp_model.fit(X_tr_s, y_tr)

perm = permutation_importance(
    imp_model,
    X_te_s,
    y_te,
    n_repeats=20,
    random_state=42,
    scoring="f1"
)

importance_df = pd.DataFrame({
    "Feature": X2.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

print(importance_df.head(15))

             Feature  Importance_Mean  Importance_Std
34   banknifty_ret_1         0.024133        0.014946
13         ICICIBANK         0.017462        0.024540
8           DOWJONES         0.011393        0.020214
33       nifty_ret_1         0.010536        0.008342
16         KOTAKBANK         0.008533        0.017659
9          NIKKEI225         0.007712        0.019454
36           usd_chg         0.006101        0.010275
37  hdfc_minus_nifty         0.004921        0.009452
5             USDINR         0.002346        0.004414
18             ret_2         0.002340        0.013732
32      range_pos_10         0.002094        0.006200
12         BRENT_OIL         0.000469        0.000878
17             ret_1         0.000081        0.001614
19             ret_3        -0.000068        0.000295
21            ret_10        -0.000477        0.015038


In [59]:
# Step 15: Keep the best features and retrain
# Why this step after Step 14:
# Step 14 showed which features matter most.
# Now we reduce noise by keeping only the top features.
# This can improve generalization because the model sees fewer weak or redundant inputs.
# We will then rerun walk-forward validation to check if performance becomes more stable.

top_features = [
    "banknifty_ret_1",
    "ICICIBANK",
    "DOWJONES",
    "nifty_ret_1",
    "KOTAKBANK",
    "NIKKEI225",
    "usd_chg",
    "hdfc_minus_nifty",
    "USDINR",
    "ret_2",
    "range_pos_10",
    "BRENT_OIL"
]

X3 = X2[top_features].copy()
y3 = y2.copy()

tscv = TimeSeriesSplit(n_splits=5)
fold_results3 = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X3), 1):
    X_tr, X_te = X3.iloc[train_idx], X3.iloc[test_idx]
    y_tr, y_te = y3.iloc[train_idx], y3.iloc[test_idx]

    fold_scaler = StandardScaler()
    X_tr_s = fold_scaler.fit_transform(X_tr)
    X_te_s = fold_scaler.transform(X_te)

    model = LogisticRegression(max_iter=2000)
    model.fit(X_tr_s, y_tr)
    pred = model.predict(X_te_s)

    fold_results3.append({
        "Fold": fold,
        "Train_Size": len(train_idx),
        "Test_Size": len(test_idx),
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred, zero_division=0),
    })

wf_results3 = pd.DataFrame(fold_results3)

print(wf_results3)
print("\nAverage metrics:")
print(wf_results3[["Accuracy", "Precision", "Recall", "F1"]].mean())

   Fold  Train_Size  Test_Size  Accuracy  Precision    Recall        F1
0     1         391        386  0.471503   0.488550  0.646465  0.556522
1     2         777        386  0.531088   0.528736  0.704082  0.603939
2     3        1163        386  0.466321   0.472222  0.083333  0.141667
3     4        1549        386  0.525907   0.591954  0.479070  0.529563
4     5        1935        386  0.474093   0.482993  0.735751  0.583162

Average metrics:
Accuracy     0.493782
Precision    0.512891
Recall       0.529740
F1           0.482970
dtype: float64


In [61]:
# Step 16: Try a Random Forest model
# Why this step after Step 15:
# We already reduced the feature list and tested logistic regression.
# Now we try a different model that can capture non-linear patterns better.
# Random Forest is a good beginner-friendly next model because it is simple to use
# and often works well on tabular data.

from sklearn.ensemble import RandomForestClassifier

# Use the selected features from Step 15
X_final = X3.copy()
y_final = y3.copy()

# Time-series walk-forward split
tscv = TimeSeriesSplit(n_splits=5)

# We will store the results from each fold here
rf_results = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_final), 1):
    # Split the data into training and testing parts for this fold
    X_train_fold = X_final.iloc[train_index]
    X_test_fold = X_final.iloc[test_index]
    y_train_fold = y_final.iloc[train_index]
    y_test_fold = y_final.iloc[test_index]

    # Scale the features using only training data
    scaler_fold = StandardScaler()
    X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
    X_test_fold_scaled = scaler_fold.transform(X_test_fold)

    # Create and train the model
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42
    )
    rf_model.fit(X_train_fold_scaled, y_train_fold)

    # Make predictions
    y_pred_fold = rf_model.predict(X_test_fold_scaled)

    # Save metrics for this fold
    rf_results.append({
        "Fold": fold,
        "Train_Size": len(train_index),
        "Test_Size": len(test_index),
        "Accuracy": accuracy_score(y_test_fold, y_pred_fold),
        "Precision": precision_score(y_test_fold, y_pred_fold, zero_division=0),
        "Recall": recall_score(y_test_fold, y_pred_fold, zero_division=0),
        "F1": f1_score(y_test_fold, y_pred_fold, zero_division=0),
    })

# Put all fold results into a table
rf_results_df = pd.DataFrame(rf_results)

print(rf_results_df)
print("\nAverage metrics:")
print(rf_results_df[["Accuracy", "Precision", "Recall", "F1"]].mean())

   Fold  Train_Size  Test_Size  Accuracy  Precision    Recall        F1
0     1         391        386  0.487047   0.500000  0.025253  0.048077
1     2         777        386  0.512953   0.517544  0.602041  0.556604
2     3        1163        386  0.494819   0.561644  0.200980  0.296029
3     4        1549        386  0.510363   0.563725  0.534884  0.548926
4     5        1935        386  0.505181   0.506410  0.409326  0.452722

Average metrics:
Accuracy     0.502073
Precision    0.529865
Recall       0.354497
F1           0.380472
dtype: float64


In [63]:
# Step 17: Compare against a simple naive baseline
# Why this step after Step 16:
# We already tried SVR, logistic regression, and random forest.
# Before we end the project, we should compare them to a very simple rule.
# This tells us whether our model is actually adding value or just matching noise.
# A naive baseline is: predict tomorrow's direction as the same as today's direction.

# Build a simple baseline using the current target
# If today's Target is 1, predict 1 for the next day
# If today's Target is 0, predict 0 for the next day
naive_pred = y3.shift(1).dropna()

# Align true values with the baseline predictions
y_true_naive = y3.loc[naive_pred.index]

print("Naive Baseline")
print("Accuracy:", round(accuracy_score(y_true_naive, naive_pred), 4))
print("Precision:", round(precision_score(y_true_naive, naive_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true_naive, naive_pred, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_true_naive, naive_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_naive, naive_pred))

print("\nClassification Report:")
print(classification_report(y_true_naive, naive_pred))

Naive Baseline
Accuracy: 0.4858
Precision: 0.5033
Recall: 0.5029
F1 Score: 0.5031

Confusion Matrix:
[[523 596]
 [597 604]]

Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.47      0.47      1119
           1       0.50      0.50      0.50      1201

    accuracy                           0.49      2320
   macro avg       0.49      0.49      0.49      2320
weighted avg       0.49      0.49      0.49      2320

